In [ ]:
# Same .env file, i know it is unsafe like this but for the sake of laziness, I am keeping it here.

HUGGINGFACE_TOKEN="YOUR_HUGGINGFACE_TOKEN"
HF_REPO_NAME="username/repo_name"

# openai api key for embedding and chat completion, langchain api key for langchain models
OPENAI_API_KEY="YOUR_OPENAI_API_KEY"
LANGCHAIN_API_KEY="LANGCHAIN_API_KEY"


In [ ]:
# download scraped files from huggingface repo to the colab, no need to run this if you are running on your local machine, since you will have the files already downloaded.
# it is recommended to run in colab, i havent tested it locally, dont blame me if it doesn't work.

!git clone https://huggingface.co/{HF_REPO_NAME}

In [ ]:
# download already saved db files from huggingface repo, just run this, don't use your brain for once

!git lfs install
!git clone https://huggingface.co/{HF_REPO_NAME}-dbfiles

In [ ]:
# move all the files and folders (that is why i told you to run this on your colab), just run it

DB_FILES = f"{HF_REPO_NAME}-dbfiles".split("/")[1]
MEOW_REPO_NAME = HF_REPO_NAME.split("/")[1]

!mkdir actual_scraped_content/
!mv {MEOW_REPO_NAME}/* ./actual_scraped_content/
!rm -rf {MEOW_REPO_NAME}
!mv {DB_FILES}/* ./
!rm -rf {DB_FILES}
!rm -rf sample_data

# for first time running, it will give some no such file or directory error, ignore it and move on further ahead

In [ ]:
# one of the reasons to run this on colab

!sudo apt-get update
!sudo apt-get install -y poppler-utils

!pip install -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

!pip install -U "unstructured[all-docs]" pillow lxml
!pip install -U chromadb tiktoken
!pip install -U langchain langchain-community langchain-openai python_dotenv
!pip install -U langchain-chroma
!pip install -U langchain-huggingface
!pip install -U transformers accelerate
!pip install -U bitsandbytes
!pip install -U unsloth

# restart runtime after installing the packages

This cell imports all the required libraries

In [ ]:
import os

from unsloth import FastLanguageModel
import glob
import uuid
import logging
import concurrent.futures
from unstructured.documents.elements import CompositeElement, Table
from unstructured.partition.auto import partition
from unstructured.partition.pdf import partition_pdf
from tqdm.notebook import tqdm
import torch
import gc
import sqlite3
import sys

from transformers import pipeline

from langchain_chroma import Chroma
from langchain.storage import LocalFileStore
from langchain.schema.document import Document
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain.retrievers.multi_vector import MultiVectorRetriever
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.messages import HumanMessage
from langchain_huggingface import HuggingFacePipeline
from google.colab import userdata

# Logging
logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s',
                    datefmt='%Y-%m-%d %H:%M:%S',
                    handlers=[logging.StreamHandler(sys.stdout)])
logger = logging.getLogger(__name__)
logging.getLogger("unstructured").setLevel(logging.ERROR)
logging.getLogger("pypdf").setLevel(logging.ERROR)
logging.getLogger("PIL").setLevel(logging.WARNING)


os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["LANGCHAIN_API_KEY"] = LANGCHAIN_API_KEY
os.environ["LANGCHAIN_TRACING_V2"] = "true" # Enable LangSmith tracing
os.environ["HF_TOKEN"] = HUGGINGFACE_TOKEN

This cell defines paths, constants, and initializes the models and retriever that will be used throughout the script.

In [ ]:
# constants
data_directory = "./actual_scraped_content/"
store_path = "./docstore/"
vectorstore_path = "./chroma_db/"
db_path = "./processed_files.db"
id_key = "doc_id"

# Create directories if they don't exist
os.makedirs(data_directory, exist_ok=True)
os.makedirs(store_path, exist_ok=True)
os.makedirs(vectorstore_path, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
logger.info(f"Using device: {device}")

# text model for summarization of tables
text_model_id = "unsloth/llama-3.1-8b-Instruct-bnb-4bit"
text_model, text_tokenizer = FastLanguageModel.from_pretrained(
    model_name=text_model_id,
    max_seq_length=2048,
    dtype=torch.float16,
    load_in_4bit=True,
)
text_pipeline = pipeline("text-generation", model=text_model, tokenizer=text_tokenizer, max_new_tokens=256)
local_text_llm = HuggingFacePipeline(pipeline=text_pipeline)

# Retriever and Vectorstore Setup
# Initialize ChromaDB client before creating the vectorstore
import chromadb
client = chromadb.PersistentClient(path=vectorstore_path)

vectorstore = Chroma(
    client=client,
    collection_name="multimodal-rag-colab",
    embedding_function=OpenAIEmbeddings(model="text-embedding-3-small", disallowed_special=()), # openai embedding model this one is good for multimodal, but paid, maybe gemini is free, if u r broke like me, use gemini
    persist_directory=vectorstore_path
)
fs = LocalFileStore(store_path)
retriever = MultiVectorRetriever(vectorstore=vectorstore, docstore=fs, id_key=id_key, search_type="similarity", retrieval_type="vectorstore")

These functions handle summarization, and document parsing. They are designed to be modular and reusable.

In [ ]:
def process_file(file_path: str, primary_summarization_llm, fallback_summarization_llm):
    """Extracts, summarizes, and stores elements from a single file with a fallback for summarization."""
    logger.info(f"Processing {file_path}...")
    try:
        file_extension = os.path.splitext(file_path)[1].lower()
        partition_params = {
            "infer_table_structure": True,
            "strategy": "auto",
            "chunking_strategy": "by_title",
            "max_characters": 2000,
            "new_after_n_chars": 1800,
            "combine_text_under_n_chars": 1000,
        }

        if file_extension == ".pdf":
            raw_elements = partition_pdf(
                filename=file_path,
                extract_images_in_pdf=False,
                **partition_params
            )
        else:
             raw_elements = partition(
                filename=file_path,
                extract_image_block_to_payload=False,
                **partition_params
            )

        # Separate elements by type
        texts, tables = [], []
        for element in raw_elements:
            if isinstance(element, Table) and element.metadata.text_as_html:
                tables.append(element)
            elif isinstance(element, CompositeElement):
                texts.append(element)

        # Prepare text docs
        text_docs = []
        text_docstore_entries = []
        if texts:
            for t in texts:
                metadata = {
                    id_key: str(uuid.uuid4()),
                    "source_file": file_path,
                    "doc_type": "text"
                }
                doc = Document(page_content=t.text, metadata=metadata)
                text_docs.append(doc)
                text_docstore_entries.append((metadata[id_key], t.text.encode('utf-8')))

        # Prepare table docs
        table_docs = []
        table_docstore_entries = []
        if tables:
            table_prompt = ChatPromptTemplate.from_template(
                "Summarize the following table to capture its key information. Table:\n{element}"
            )
            try:
                # Attempt to use the primary (fast) summarization model
                logger.info(f"Summarizing {len(tables)} tables with primary model for {os.path.basename(file_path)}...")
                primary_chain = {"element": lambda t: t.metadata.text_as_html} | table_prompt | primary_summarization_llm | StrOutputParser()
                table_summaries = primary_chain.batch(tables, {"max_concurrency": 8})
            except Exception as e:
                logger.warning(f"Primary summarization failed for {os.path.basename(file_path)}: {e}. Switching to fallback model.")
                # If it fails, use the fallback (local) model
                fallback_chain = {"element": lambda t: t.metadata.text_as_html} | table_prompt | fallback_summarization_llm | StrOutputParser()
                table_summaries = fallback_chain.batch(tables, {"max_concurrency": 2}) # Lower concurrency for local model

            for i, s in enumerate(table_summaries):
                metadata = {
                    id_key: str(uuid.uuid4()),
                    "source_file": file_path,
                    "doc_type": "table"
                }
                doc = Document(page_content=s, metadata=metadata)
                table_docs.append(doc)
                table_docstore_entries.append((metadata[id_key], tables[i].metadata.text_as_html.encode('utf-8')))

        # Only write to vectorstore and docstore if all succeeded
        if text_docs:
            retriever.vectorstore.add_documents(text_docs)
            retriever.docstore.mset(text_docstore_entries)
            logger.info(f"Added {len(text_docs)} text chunks for {os.path.basename(file_path)}")
        if table_docs:
            retriever.vectorstore.add_documents(table_docs)
            retriever.docstore.mset(table_docstore_entries)
            logger.info(f"Added {len(table_docs)} table summaries for {os.path.basename(file_path)}")

        return file_path
    except Exception as e:
        logger.error(f"Failed to process {file_path}: {e}", exc_info=True)
        return None

def setup_database(db_path: str):
    """Initializes the SQLite database to track processed and useless files."""
    with sqlite3.connect(db_path) as conn:
        cursor = conn.cursor()
        cursor.execute("CREATE TABLE IF NOT EXISTS processed_files (path TEXT PRIMARY KEY)")
        cursor.execute("CREATE TABLE IF NOT EXISTS useless_files (path TEXT PRIMARY KEY)")
        conn.commit()
    return conn

def get_processed_files(conn):
    """Retrieves the set of already processed file paths from the database."""
    cursor = conn.cursor()
    cursor.execute("SELECT path FROM processed_files")
    return {row[0] for row in cursor.fetchall()}

def get_useless_files(conn):
    """Retrieves the set of file paths which are not to be processed from the database."""
    cursor = conn.cursor()
    cursor.execute("SELECT path FROM useless_files")
    return {row[0] for row in cursor.fetchall()}

def add_processed_file(conn, file_path: str):
    """Adds a newly processed file path to the database."""
    cursor = conn.cursor()
    cursor.execute("INSERT OR IGNORE INTO processed_files (path) VALUES (?)", (file_path,))
    conn.commit()

def add_useless_file(conn, file_path: str):
    """Adds file path to the database which we do not have to process."""
    cursor = conn.cursor()
    cursor.execute("INSERT OR IGNORE INTO useless_files (path) VALUES (?)", (file_path,))
    conn.commit()

This section orchestrates the file processing. It scans the data directory, filters out already processed files, and then processes new ones sequentially.

In [ ]:
import os
import glob
import gc
import signal

class TimeoutError(Exception):
    pass

def timeout_handler(signum, frame):
    """This function is called when the alarm signal is received."""
    raise TimeoutError("TIMEOUT: Processing exceeded 5 minutes and was skipped.")

signal.signal(signal.SIGALRM, timeout_handler)


def main_ingestion_sequential_with_timeout():
    db_conn = setup_database(db_path)
    processed_files = get_processed_files(db_conn)
    useless_files = get_useless_files(db_conn)
    logger.info(f"Found {len(processed_files)} previously processed files.")
    logger.info(f"Found {len(useless_files)} useless files.")

    all_files = []
    for ext in ["*.pdf", "*.docx", "*.txt"]: # you can add more extensions if you want to process them as well but don't blame me
        all_files.extend(glob.glob(os.path.join(data_directory, f"**/{ext}"), recursive=True))

    files_to_process = sorted([f for f in all_files if f not in processed_files and f not in useless_files])

    if not files_to_process:
        logger.warning("No new files to process.")
        db_conn.close()
        return

    logger.info(f"Found {len(files_to_process)} new files to process.")

    primary_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, max_tokens=512)

    with tqdm(total=len(files_to_process), desc="Processing files sequentially") as pbar:
        for file_path in files_to_process:
            base_name = os.path.basename(file_path)
            pbar.set_postfix_str(f"Current: {base_name}", refresh=True)

            try:
                signal.alarm(300)  # 300 seconds = 5 minutes, if the processing takes longer, it will raise a TimeoutError

                result = process_file(file_path, primary_llm, local_text_llm)

                if result:
                    add_processed_file(db_conn, result)
                    logger.info(f"Successfully processed and recorded: {base_name}")
                else:
                    # If process_file returns None, it means an error occurred, add to useless files
                    add_useless_file(db_conn, file_path)
                    logger.warning(f"Skipped {base_name} due to processing failure (returned None). Added to useless files.")

            except TimeoutError as e:
                # This block will execute if the signal handler raises the TimeoutError
                add_useless_file(db_conn, file_path) # Add to useless files on timeout
                logger.error(f"Timeout processing {base_name}. Skipping. Added to useless files. Details: {e}")
            except Exception as e:
                # Catches any other error during file processing
                add_useless_file(db_conn, file_path) # Add to useless files on other errors
                logger.error(f"FATAL: An error occurred while processing {base_name}: {e}")
            finally:
                # This prevents the alarm from firing later if the function finished on time.
                signal.alarm(0)
                pbar.update(1)
                gc.collect()

    db_conn.close()
    logger.info("File ingestion complete.")

if __name__ == "__main__":
    main_ingestion_sequential_with_timeout()

The final step defines the complete RAG chain and allows you to ask questions. The chain retrieves relevant context (text, tables), constructs a multi-modal prompt, and sends it to the `gpt-4o-mini` model for a final answer.

In [ ]:
def parse_retrieved_docs(docs):
    """
    Parses retrieved Document objects, loading original content from the docstore
    and categorizing them by type (text, table).
    """
    parsed_texts = []
    parsed_tables = []

    # 'docs' is a list of Document objects returned by the retriever.
    # Each Document has page_content (the embedded summary/chunk) and metadata.
    # We use the 'doc_id' from metadata to fetch the original content from docstore.
    doc_ids_to_fetch = [doc.metadata[id_key] for doc in docs if id_key in doc.metadata]

    if not doc_ids_to_fetch:
        logger.warning("No document IDs found in retrieved documents to fetch from docstore.")
        return {"texts": [], "tables": []}

    try:
        # mget returns a list of contents corresponding to the input list of IDs
        contents_bytes = retriever.docstore.mget(doc_ids_to_fetch)

        for i, content_bytes in enumerate(contents_bytes):
            if content_bytes is None:
                logger.warning(f"Content for doc_id {doc_ids_to_fetch[i]} not found in docstore. Skipping.")
                continue

            try:
                content = content_bytes.decode('utf-8')
                # Use the 'doc_type' metadata we added previously for reliable categorization
                doc_type = docs[i].metadata.get("doc_type", "unknown")
                source_file = docs[i].metadata.get("source_file", "N/A")

                if doc_type == "table":
                    # For tables, include a clear label that it's a table and its source
                    parsed_tables.append(f"--- Table from {os.path.basename(source_file)} ---\n{content}\n--- End Table ---")
                elif doc_type == "text":
                    # For text, just append the content, optionally with source info
                    parsed_texts.append(f"--- Text from {os.path.basename(source_file)} ---\n{content}\n--- End Text ---")
                else:
                    logger.warning(f"Document with unknown doc_type '{doc_type}' from {source_file}. Appending as text.")
                    parsed_texts.append(f"--- Unknown Document Type from {os.path.basename(source_file)} ---\n{content}\n--- End Unknown Type ---")

            except UnicodeDecodeError:
                logger.error(f"Could not decode content for doc_id {doc_ids_to_fetch[i]} from {docs[i].metadata.get('source_file', 'N/A')}. Skipping.")
    except Exception as e:
        logger.error(f"Error processing doc_id {doc_ids_to_fetch[i]} from {docs[i].metadata.get('source_file', 'N/A')}: {e}", exc_info=True)

    return {"texts": parsed_texts, "tables": parsed_tables}


def build_prompt(inputs):
    """
    Builds a multi-modal prompt from structured context (texts, tables) and a question.
    Formats the context clearly for the LLM.
    """
    context_parts = []

    if inputs["context"].get("texts"):
        context_parts.append("--- Retrieved Text Content ---")
        context_parts.extend(inputs["context"]["texts"])

    if inputs["context"].get("tables"):
        if context_parts:
            context_parts.append("\n")
        context_parts.append("--- Retrieved Table Content (as HTML) ---")
        context_parts.extend(inputs["context"]["tables"])


    context_text = "\n\n".join(context_parts)

    if not context_text.strip():
        # Handle cases where no relevant context is found
        prompt_template = (
            f"No relevant context was found for the question: '{inputs['question']}'. "
            "Please state that you cannot answer based on the provided information."
        )
    else:
        prompt_template = (
            f"Answer the question based ONLY on the provided context. "
            f"The context may include plain text and tables represented as HTML. "
            f"Summarize tables concisely if they are relevant to the question. "
            f"If the answer is not in the context, state that you cannot answer.\n\n"
            f"Context:\n{context_text}\n\n"
            f"Question: {inputs['question']}"
        )

    prompt_content = [{"type": "text", "text": prompt_template}]
    return [HumanMessage(content=prompt_content)]

# --- Final RAG Chain ---
final_model = ChatOpenAI(model="gpt-4o-mini", temperature=0.2, max_tokens=1024)
rag_chain = (
    {
        "context": retriever.vectorstore.as_retriever() | RunnableLambda(parse_retrieved_docs),
        "question": RunnablePassthrough()
    }
    | RunnableLambda(build_prompt)
    | final_model
    | StrOutputParser()
)

logger.info("\n--- RAG Pipeline Ready for Query ---")

# --- Example Query ---
question = "What does the document say about? Summarize the key points."
logger.info(f"Querying with: '{question}'")

# Add this before the RAG chain invocation for debugging
docs = retriever.vectorstore.as_retriever().invoke(question)
# Check if docs is empty before accessing docs[0]
if docs:
    print("Type of first retrieved doc:", type(docs[0]))

# Invoke the chain and print the response
response = rag_chain.invoke(question)

print("\n--- Response ---")
print(response)

In [ ]:
# code to upload all processed db files and folders to huggingface repo


from huggingface_hub import HfApi, CommitOperationAdd

repo_id = f"{HF_REPO_NAME}-dbfiles"
token = HUGGINGFACE_TOKEN

api = HfApi()

# List all files and directories to upload (exclude specific files if needed)
files_to_upload = [os.path.join(root, file)
                   for root, _, files in os.walk('.')
                   for file in files
                   if not (root.startswith('./actual_scraped_content') or root.startswith('./sample_data'))]

operations = [
    CommitOperationAdd(file, file)
    for file in files_to_upload
]

try:
    api.create_commit(
        repo_id=repo_id,
        operations=operations,
        commit_message="Upload all project files and folders",
        token=token,
        repo_type="model"
    )
    logging.info(f"Successfully uploaded files to Hugging Face repo: {repo_id}")
except Exception as e:
    logging.error(f"Failed to upload files to Hugging Face: {e}")
    logging.warning("Please ensure the repository exists and your HF_TOKEN has write access.")

After running all these, go to readme.md for step 4.